# CASCAQit read-only renderers

This offline notebook exercises the Digital, Analog, Result, Diagnostics, and Visualization MIME views.

In [1]:
# 导入构造 Digital/Analog 程序和 counts 直方图所需的公开 API。
from cascaqit import AHSProgram, AtomRegister, Circuit, Waveform, build_counts_histogram
# DiagnosticsIR 用于构造可交给 Diagnostics renderer 的结构化诊断。
from cascaqit.diagnostics import DiagnosticsIR
# 导入原子阵列和脉冲时间线两种可视化构造器。
from cascaqit.visualization import build_pulse_timeline, build_register_visualization
# IPython.display.display 负责把富 MIME 对象发送到 Notebook 输出区。
from IPython.display import display

# 导入 CASCAQit-Jupyter 的四类只读 renderer 入口。
from cascaqit_jupyter import (
    # 把 DiagnosticsIR 转成诊断列表输出。
    display_diagnostics,
    # 把 Digital 或 Analog ProgramIR 转成程序视图。
    display_program,
    # 把 ResultIR 转成 shots、counts、概率和诊断视图。
    display_result,
    # 把 VisualizationIR 转成直方图、阵列或时间线。
    display_visualization,
)

# 创建两个量子比特的 Digital 线路，并设置稳定 program_id。
digital_circuit = Circuit(2, program_id="program.e2e.bell")
# q0 经过 H 门后与 q1 建立纠缠，最后测量全部量子比特。
digital_circuit.h(0).cx(0, 1).measure_all()
# 转为 ProgramIR，稍后交给 Program renderer 显示。
digital_program = digital_circuit.to_program()
# 在本地运行 32 shots；固定 seed，并同时返回 probabilities。
result = digital_circuit.run(shots=32, seed=2026, return_probabilities=True)

# 创建四位点 Analog 程序；builder 保留链式构造接口。
analog_builder = AHSProgram(
    # 四个原子沿直线排列，间距为 5.0，并设置稳定 program_id。
    AtomRegister.line(count=4, spacing=5.0), program_id="program.e2e.analog"
# 添加全局 Rabi、detuning 和 phase 驱动。
).drive(
    # Rabi 在四个时间点上升、保持再回落。
    rabi=Waveform.piecewise_linear(
        # times 和 values 一一对应；waveform_id 用于报告和诊断定位。
        times=(0.0, 0.4, 0.8, 1.2), values=(0.0, 2.5, 2.5, 0.0), waveform_id="rabi"
    ),
    # Detuning 在 1.2 时长内从 -4.0 线性变化到 4.0。
    detuning=Waveform.linear(-4.0, 4.0, duration=1.2, waveform_id="detuning"),
    # 本例使用固定全局相位 0.0。
    phase=0.0,
# 在 Analog 程序末尾添加 measurement_id 为 m 的测量。
).measure(measurement_id="m")
# 转为 Analog ProgramIR，供程序、阵列和脉冲 renderer 共同使用。
analog_program = analog_builder.to_ir()

# 构造一条带不可信文本的 DiagnosticsIR，用于验证 renderer 会转义 HTML。
diagnostic = DiagnosticsIR(
    # diagnostic_id 是该诊断的稳定标识。
    diagnostic_id="diagnostic.e2e.untrusted",
    # stage 表示问题在 validation 阶段产生。
    stage="validation",
    # severity=error 让 renderer 使用错误级别样式。
    severity="error",
    # code 是程序可读取的稳定诊断码。
    code="E2E_UNTRUSTED_TEXT",
    # message 故意包含 HTML 和 script，确认它们只作为文本显示。
    message=(
        '<img src=x onerror="window.__cascaqitXss=true">'
        "<script>window.__cascaqitXss=true</script>"
    ),
    # object_path 指向诊断关联的第一条门指令。
    object_path="circuit.gates[0]",
    # suggestion 告诉使用者必须把 message 当作纯文本。
    suggestion="Treat this value as text only.",
)

In [2]:
# 显示 Digital ProgramIR；这一行不会重新运行线路。
display(display_program(digital_program))
# 显示 Analog ProgramIR，包括原子阵列和全局控制摘要。
display(display_program(analog_program))
# 显示已有 ResultIR 中的 shots、bit order、counts、概率和诊断。
display(display_result(result))
# 显示结构化诊断，并用 source_id 关联到 Bell 程序。
display(display_diagnostics(diagnostic, source_id="program.e2e.bell"))
# 从已有 ResultIR 构造 counts 直方图，再交给 Visualization renderer。
display(display_visualization(build_counts_histogram(result)))
# 从 Analog ProgramIR 构造原子阵列图，不访问后端。
display(display_visualization(build_register_visualization(analog_program)))
# 从 Analog ProgramIR 构造脉冲时间线，不修改原始波形。
display(display_visualization(build_pulse_timeline(analog_program)))

CASCAQit program: program.e2e.bell [4bc3dc99d3f1]

CASCAQit program: program.e2e.analog [76ad692448a0]

CASCAQit result: result.program.e2e.bell [4381e04adaae]

CASCAQit diagnostics: program.e2e.bell [23f92b5de587]

CASCAQit visualization: viz.counts.result.program.e2e.bell [2b6877fd7783]

CASCAQit visualization: viz.register.program.e2e.analog [38b8fa9d775b]

CASCAQit visualization: viz.pulse_timeline.program.e2e.analog [ad9cf4b6d1db]